# RXN2 ChEMBL 37 catalogue export
Run each cell in order. Raw archive and compact outputs stay in Drive; extraction is ephemeral in Colab.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, re, sqlite3, tarfile, urllib.request
from datetime import UTC, datetime

drive.mount('/content/drive')
RAW = Path('/content/drive/MyDrive/RXN2/data/raw/chembl/chembl_37')
OUT = Path('/content/drive/MyDrive/RXN2/data/processed/chembl/chembl_37')
MANIFEST = Path('/content/drive/MyDrive/RXN2/data/manifests/chembl-37.json')
WORK = Path('/content/rxn2_chembl37')
for path in (RAW, OUT, MANIFEST.parent, WORK): path.mkdir(parents=True, exist_ok=True)
BASE = 'https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/releases/chembl_37'
ARCHIVE = RAW / 'chembl_37_sqlite.tar.gz'

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()

checksums = urllib.request.urlopen(BASE + '/checksums.txt').read().decode('utf-8')
match = re.search(r'([0-9a-fA-F]{64})\s+\*?chembl_37_sqlite\.tar\.gz', checksums)
if not match: raise RuntimeError('missing ChEMBL 37 archive checksum')
expected = match.group(1).lower()
if not ARCHIVE.exists() or sha256(ARCHIVE) != expected:
    partial = ARCHIVE.with_suffix('.tar.gz.partial')
    partial.unlink(missing_ok=True)
    with urllib.request.urlopen(BASE + '/chembl_37_sqlite.tar.gz') as source, partial.open('wb') as output:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''): output.write(chunk)
    partial.replace(ARCHIVE)
actual = sha256(ARCHIVE)
if actual != expected: raise RuntimeError(f'checksum mismatch: {actual}')
print({'archive': str(ARCHIVE), 'bytes': ARCHIVE.stat().st_size, 'sha256': actual})


In [ ]:
with tarfile.open(ARCHIVE, 'r:gz') as tar:
    matches = [member for member in tar.getmembers() if member.name.endswith('/chembl_37.db')]
    if len(matches) != 1: raise RuntimeError(f'expected one database, found {len(matches)}')
    member = matches[0]
    CHEMBL_DB = (WORK / member.name).resolve()
    if WORK.resolve() not in CHEMBL_DB.parents: raise RuntimeError('unsafe archive path')
    if not CHEMBL_DB.exists(): tar.extract(member, WORK)
print({'database': str(CHEMBL_DB), 'bytes': CHEMBL_DB.stat().st_size})


In [ ]:
JSONL, REPORT = OUT / 'catalogue.jsonl', OUT / 'report.json'
db = sqlite3.connect(f'file:{CHEMBL_DB.as_posix()}?mode=ro', uri=True)
db.row_factory = sqlite3.Row
required = {'molecule_dictionary', 'compound_structures', 'molecule_hierarchy', 'molecule_synonyms'}
present = {row[0] for row in db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
if required - present: raise RuntimeError(f'missing ChEMBL tables: {sorted(required - present)}')
aliases = {}
for row in db.execute("SELECT s.molregno, s.synonyms, s.syn_type FROM molecule_synonyms s JOIN molecule_dictionary md USING (molregno) WHERE md.max_phase=4 AND lower(md.molecule_type)='small molecule'"):
    if row['synonyms']: aliases.setdefault(int(row['molregno']), []).append({'value': str(row['synonyms']).strip(), 'type': str(row['syn_type'] or 'synonym').strip()})
query = """SELECT md.molregno, md.chembl_id, md.pref_name, cs.canonical_smiles, cs.standard_inchi, cs.standard_inchi_key, COALESCE(mh.parent_molregno, md.molregno) parent_molregno, COALESCE(parent.chembl_id, md.chembl_id) parent_chembl_id, COALESCE(parent.pref_name, md.pref_name, md.chembl_id) parent_name FROM molecule_dictionary md LEFT JOIN compound_structures cs ON cs.molregno=md.molregno LEFT JOIN molecule_hierarchy mh ON mh.molregno=md.molregno LEFT JOIN molecule_dictionary parent ON parent.molregno=COALESCE(mh.parent_molregno,md.molregno) WHERE md.max_phase=4 AND lower(md.molecule_type)='small molecule' ORDER BY COALESCE(mh.parent_molregno,md.molregno), md.molregno != COALESCE(mh.parent_molregno,md.molregno), md.molregno"""
partial, count = JSONL.with_suffix('.jsonl.partial'), 0
with partial.open('w', encoding='utf-8', newline='\n') as output:
    for row in db.execute(query):
        parent = row['parent_chembl_id']
        record = {'preferred_name': row['parent_name'], 'aliases': aliases.get(int(row['molregno']), []), 'identifiers': {'CHEMBL': parent}, 'active_moiety_id': f'chembl-moiety:{parent}', 'compound': {'compound_id': row['chembl_id'], 'smiles': row['canonical_smiles'], 'inchi': row['standard_inchi'], 'inchi_key': row['standard_inchi_key'], 'material_form': 'active_moiety' if row['chembl_id']==parent else 'salt_or_form', 'relationship_type': 'active_moiety' if row['chembl_id']==parent else 'salt_or_form'}}
        output.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + '\n'); count += 1
partial.replace(JSONL); db.close()
result = {'contract': 'rxn2-cloud-result-v1', 'command': 'chembl', 'release': 'ChEMBL37', 'released_on': '2026-05-29', 'provider_url': BASE, 'license': 'CC BY-SA 3.0', 'created_at': datetime.now(UTC).isoformat(), 'source_archive': {'path': str(ARCHIVE), 'size_bytes': ARCHIVE.stat().st_size, 'sha256': actual}, 'output': str(JSONL), 'output_size_bytes': JSONL.stat().st_size, 'output_sha256': sha256(JSONL), 'catalogue_records': count}
REPORT.write_text(json.dumps(result, indent=2, sort_keys=True) + '\n', encoding='utf-8')
MANIFEST.write_text(json.dumps({'manifest_version':'1.0.0','source_id':'chembl_snapshot','release_id':'ChEMBL37','released_on':'2026-05-29','license':'CC BY-SA 3.0','provider_url':BASE,'entries':[{'path':ARCHIVE.name,'size_bytes':ARCHIVE.stat().st_size,'sha256':actual}]}, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(result)
